In [ ]:
# Instalacion de LazyPredict (primera ejecucion en Colab)
# !pip install lazypredict


In [ ]:
# =================================================================
# PROYECTO INTEGRADOR: MACHINE LEARNING - FASE 2B
# GRUPO 2: Analisis de Satisfaccion en Productos de Oficina
# Notebook: AutoML Benchmark (LazyPredict)
# =================================================================

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import gc
import json
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules

# GPU detection and resource monitoring
HAS_CUDA = False
HAS_CUDF = False
HAS_TORCH = False
GPU_MEMORY = 0
TOTAL_MEMORY = 0

try:
    import torch
    HAS_TORCH = True
    HAS_CUDA = torch.cuda.is_available()
    if HAS_CUDA:
        GPU_MEMORY = torch.cuda.get_device_properties(0).total_memory
except ImportError:
    pass

try:
    import cudf
    HAS_CUDF = True
except ImportError:
    pass

try:
    import psutil
    TOTAL_MEMORY = psutil.virtual_memory().total
except ImportError:
    pass

# Progress tracking
from tqdm.auto import tqdm

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/ML/proyecto_integrador')
else:
    BASE = Path('..')

DATA_DIR = BASE / 'data'
REPORTS_DIR = BASE / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Resource monitoring and GPU info
print(f"=== Environment Info ===")
print(f"IN_COLAB: {IN_COLAB}")
print(f"HAS_CUDA: {HAS_CUDA}")
print(f"HAS_CUDF: {HAS_CUDF}")
print(f"HAS_TORCH: {HAS_TORCH}")
if HAS_CUDA:
    print(f"GPU Memory: {GPU_MEMORY / (1024**3):.1f} GB")
if TOTAL_MEMORY:
    print(f"System RAM: {TOTAL_MEMORY / (1024**3):.1f} GB")
print(f"BASE: {BASE}")
print(f"DATA_DIR: {DATA_DIR}")

from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_sample_weight


In [ ]:
print('Cargando dataset canonico desde f1_eda_definitivo...')

CANONICAL_PATH = DATA_DIR / 'office_products_balanced.parquet'

try:
    df = pd.read_parquet(CANONICAL_PATH, columns=['text', 'rating', 'sentiment'])
except FileNotFoundError:
    print(f'[ERROR] Dataset canonico no encontrado en {CANONICAL_PATH}')
    print('Ejecute primero f1_eda_definitivo.ipynb en Colab para generarlo.')
    raise
except Exception as e:
    print(f'[ERROR] No se pudo cargar el dataset: {e}')
    raise

if 'sentiment' not in df.columns:
    df['sentiment'] = df['rating'].map(lambda r: 0 if r <= 2 else (1 if r == 3 else 2))

# word_count filter
df['word_count'] = df['text'].astype(str).str.split().str.len()
df = df[df['word_count'] >= 5]
print(f'Registros tras filtro word_count >= 5: {len(df):,}')
print('Distribucion de sentimiento:')
print(df['sentiment'].value_counts().sort_index())


In [ ]:
# Misma particion 80/20 estratificada para comparacion justa
# Nota: Para LazyPredict reducimos a 50k muestras (limite RAM con matrices densas)
print('Creando particion estratificada 80/20...')

SAMPLE_SIZE = 50000
df_sample = df.groupby('sentiment', group_keys=False).apply(
    lambda x: x.sample(n=min(SAMPLE_SIZE // 3, len(x)), random_state=42)
).reset_index(drop=True)

print(f'Muestra estratificada: {len(df_sample):,} filas')
print(df_sample['sentiment'].value_counts().sort_index())

del df
gc.collect()


In [ ]:
print('Vectorizando con TF-IDF...')
vectorizer = TfidfVectorizer(max_features=1500, stop_words='english', ngram_range=(1, 2), min_df=5)
X_sparse = vectorizer.fit_transform(df_sample['text'].astype(str))
y = df_sample['sentiment'].values
print(f'Matriz TF-IDF: {X_sparse.shape}')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_sparse, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')


In [ ]:
print('\nEjecutando LazyPredict (27+ clasificadores)...')
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None, class_weight='balanced')
models_df, predictions = clf.fit(X_train, X_test, y_train, y_test)
gc.collect()

print('\nTop 10 modelos segun LazyPredict:')
print(models_df.head(10).to_string())


In [ ]:
results_path = REPORTS_DIR / 'lazypredict_results.csv'
models_df.to_csv(results_path)
print(f'Resultados guardados en {results_path}')


In [ ]:
top = models_df.head(10).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(top)), top['F1 Score'], color='steelblue')
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top['Model'])
ax.set_xlabel('F1 Score')
ax.set_title('Top 10 Modelos - LazyPredict (F1-Score)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Modelo explicativo con class_weight
modelo_explicativo = LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)
modelo_explicativo.fit(X_train, y_train)

# Evaluacion
y_pred_lgb = modelo_explicativo.predict(X_test)
f1 = f1_score(y_test, y_pred_lgb, average='macro')
print(f'LightGBM F1-macro: {f1:.4f}')


In [ ]:
feature_names = vectorizer.get_feature_names_out()


In [ ]:
importances = modelo_explicativo.feature_importances_
df_importances = pd.DataFrame({
    'Caracteristica': feature_names,
    'Importancia': importances
}).sort_values(by='Importancia', ascending=False).head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(df_importances)), df_importances['Importancia'].values, color='coral')
ax.set_yticks(range(len(df_importances)))
ax.set_yticklabels(df_importances['Caracteristica'].values)
ax.set_xlabel('Importancia')
ax.set_title('Top 20 Features - LightGBM')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
print('\nBenchmark AutoML completado.')
print(f'Resultados: lazypredict_results.csv')
print('Compare estos resultados con las metricas en metrics_fase2.json')
